In [ ]:
# =============================================
# SECTION 1  — 3.1 Dataset Requirements
# =============================================
# Requirements covered:
# - Dataset >= 10,000 users, >= 500 items, >= 100,000 ratings
# - Ratings on 1–5 scale 
# - Select target users (U1,U2,U3): cold <=2%, medium 2–5%, rich >10%
# - Select target items (I1,I2) based on popularity: low + high popularity
# - Save preprocessing outputs for later parts
# - Round numerical values to 2 decimals in outputs

import os
import numpy as np
import pandas as pd

# --- Path ---
from pathlib import Path

PROJECT_ROOT = Path.cwd()

def resolve_existing(*candidates: str) -> str:
    for c in candidates:
        p = (PROJECT_ROOT / c)
        if p.exists():
            return str(p)
    tried = "\n".join([f"- {PROJECT_ROOT / c}" for c in candidates])
    raise FileNotFoundError(f"Could not find required file. Tried:\n{tried}")

# Common dataset candidates 
RATINGS_PATH = resolve_existing(
    r"C:\ml-20m\ml-20m\ratings.csv",)
    


np.random.seed(42)

# ----- Paths -----
# DATASET path is resolved via RATINGS_PATH helper above
OUT_DIR = "SECTION1_DimensionalityReduction/data"
os.makedirs(OUT_DIR, exist_ok=True)

# ----- Load ratings -----
ratings = pd.read_csv(RATINGS_PATH, usecols=["userId", "movieId", "rating"])

# ----- Ensure ratings are on 1–5 scale -----
# MovieLens is already within [0.5, 5.0]. We clip to enforce 1–5.
ratings["rating"] = ratings["rating"].clip(lower=1, upper=5)


# ratings["rating"] = ratings["rating"].round().astype(int)

# ----- Basic dataset stats -----
num_users = ratings["userId"].nunique()
num_items = ratings["movieId"].nunique()
num_ratings = len(ratings)

print(f"Users: {num_users:,}")
print(f"Items: {num_items:,}")
print(f"Ratings: {num_ratings:,}")

# ----- Enforce minimum dataset requirements -----
assert num_users >= 10_000, "Dataset requirement NOT met: users < 10,000"
assert num_items >= 500, "Dataset requirement NOT met: items < 500"
assert num_ratings >= 100_000, "Dataset requirement NOT met: ratings < 100,000"

# =============================================
# User stats: n_u, % items rated per user
# =============================================
user_counts = ratings.groupby("userId")["movieId"].count().rename("n_u").reset_index()
user_counts["pct_items_rated"] = (user_counts["n_u"] / num_items * 100).round(2)

# Save user stats
user_counts.to_csv(os.path.join(OUT_DIR, "user_stats.csv"), index=False)

# =============================================
# Item stats: n_i, % users who rated the item 
# =============================================
item_counts = ratings.groupby("movieId")["userId"].count().rename("n_i").reset_index()
item_counts["popularity_pct"] = (item_counts["n_i"] / num_users * 100).round(2)

# Save item stats
item_counts.to_csv(os.path.join(OUT_DIR, "item_stats.csv"), index=False)

# =============================================
# Select Target Users U1, U2, U3 
# =============================================
cold_pool   = user_counts[user_counts["pct_items_rated"] <= 2].copy()
medium_pool = user_counts[(user_counts["pct_items_rated"] >= 2) & (user_counts["pct_items_rated"] <= 5)].copy()
rich_pool   = user_counts[user_counts["pct_items_rated"] > 10].copy()  

assert len(cold_pool) > 0, "No cold users found with <=2% ratings"
assert len(medium_pool) > 0, "No medium users found with 2–5% ratings"
assert len(rich_pool) > 0, "No rich users found with >10% ratings"

U1 = int(cold_pool.sample(1, random_state=42)["userId"].iloc[0])
U2 = int(medium_pool.sample(1, random_state=42)["userId"].iloc[0])
U3 = int(rich_pool.sample(1, random_state=42)["userId"].iloc[0])

targets_users = pd.DataFrame({
    "user_label": ["U1_cold", "U2_medium", "U3_rich"],
    "userId": [U1, U2, U3]
})

print("\nTarget Users (U1, U2, U3):")
display(targets_users)

targets_users.to_csv(os.path.join(OUT_DIR, "target_users.csv"), index=False)

# =============================================
# Select Target Items I1, I2 
# Choose: I1 = low popularity, I2 = high popularity
# =============================================

# Define popularity pools 
low_item_pool  = item_counts[item_counts["popularity_pct"] <= 1].copy()   # low popularity (tail)
high_item_pool = item_counts[item_counts["popularity_pct"] >= 10].copy()  # high popularity (head)

# If thresholds are too strict for your dataset, fallback to quantiles
if len(low_item_pool) == 0:
    q10 = item_counts["popularity_pct"].quantile(0.10)
    low_item_pool = item_counts[item_counts["popularity_pct"] <= q10].copy()

if len(high_item_pool) == 0:
    q90 = item_counts["popularity_pct"].quantile(0.90)
    high_item_pool = item_counts[item_counts["popularity_pct"] >= q90].copy()

I1 = int(low_item_pool.sample(1, random_state=42)["movieId"].iloc[0])
I2 = int(high_item_pool.sample(1, random_state=42)["movieId"].iloc[0])

targets_items = pd.DataFrame({
    "item_label": ["I1_low_popularity", "I2_high_popularity"],
    "movieId": [I1, I2]
})

print("\nTarget Items (I1, I2):")
display(targets_items)

targets_items.to_csv(os.path.join(OUT_DIR, "target_items.csv"), index=False)

# =============================================
# Quick verification prints (2 decimals)
# =============================================
u_check = user_counts.set_index("userId").loc[[U1, U2, U3]].copy()
i_check = item_counts.set_index("movieId").loc[[I1, I2]].copy()

print("\nSelected Users Check:")
display(u_check)

print("\nSelected Items Check:")
display(i_check)



Users: 138,493
Items: 26,744
Ratings: 20,000,263

Target Users (U1, U2, U3):


,user_label,userId
0,U1_cold,121238
1,U2_medium,43905
2,U3_rich,68026



Target Items (I1, I2):


,item_label,movieId
0,I1_low_popularity,118758
1,I2_high_popularity,235



Selected Users Check:


,n_u,pct_items_rated
userId,,
121238,133,0.50
43905,564,2.11
68026,3602,13.47



Selected Items Check:


,n_i,popularity_pct
movieId,,
118758,1,0.00
235,16419,11.86
